# MixedSingleTaskGP

`robotorchan.models.MixedSingleTaskGP` を用いて、**連続変数 + カテゴリ変数**の混合探索空間をモデル化します。

このNotebookでは、混合データの作成、`cat_dims` を指定したモデル構築、robotorchan共通API、カテゴリ別事後分布、可視化、`optimize_acqf_mixed` による候補点選択までを扱います。


In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.optim.optimize import optimize_acqf_mixed

from robotorchan.models import MixedSingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 混合型の合成データ

入力は2次元です。

- `x[:, 0]`: 連続変数 (`0 <= x <= 1`)
- `x[:, 1]`: カテゴリ変数 (`0, 1, 2`)

カテゴリごとに目的関数のオフセットと形状を少し変えます。


In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    x = X[..., 0]
    category = X[..., 1]

    base = torch.sin(2 * torch.pi * x)
    category_effect = torch.where(
        category == 0,
        torch.zeros_like(x),
        torch.where(
            category == 1,
            0.45 + 0.15 * torch.cos(4 * torch.pi * x),
            -0.25 + 0.25 * x,
        ),
    )
    return (base + category_effect).unsqueeze(-1)

x_cont = torch.linspace(0.05, 0.95, 8)

train_X = torch.cat(
    [
        torch.stack([x_cont, torch.full_like(x_cont, category)], dim=-1)
        for category in (0.0, 1.0, 2.0)
    ],
    dim=0,
)
train_Y = objective(train_X)

print("train_X:", train_X.shape)
print("train_Y:", train_Y.shape)
print("categories:", train_X[:, 1].unique())


## 2. モデル構築

2列目 (`index=1`) をカテゴリ次元として `cat_dims=[1]` で指定します。

カテゴリ値 `0, 1, 2` はカテゴリラベルであり、順序や距離を持つ連続値として扱うためのものではありません。


In [ ]:
model = MixedSingleTaskGP(
    train_X=train_X,
    train_Y=train_Y,
    cat_dims=[1],
)

print(type(model).__name__)
print("supports_mll:", model.supports_mll)


## 3. robotorchan共通API


In [ ]:
print("raw_data_names:", model.raw_data_names)
print("raw_train_X shape:", model.raw_train_X.shape)
print("raw_train_Y shape:", model.raw_train_Y.shape)
print("raw_train_Yvar:", model.raw_train_Yvar)


## 4. モデル学習


In [ ]:
mll = model.make_mll()
print(type(mll).__name__)

fit_gpytorch_mll(mll)
model.eval()


## 5. カテゴリ別の事後分布予測

連続変数のグリッドを共通にし、カテゴリ値だけを固定して事後分布を比較します。


In [ ]:
grid = torch.linspace(0.0, 1.0, 201)

posterior_by_category = {}

with torch.no_grad():
    for category in (0.0, 1.0, 2.0):
        test_X = torch.stack(
            [grid, torch.full_like(grid, category)],
            dim=-1,
        )
        posterior = model.posterior(test_X)
        mean = posterior.mean.squeeze(-1)
        lower, upper = posterior.mvn.confidence_region()

        posterior_by_category[int(category)] = {
            "mean": mean,
            "lower": lower.squeeze(-1),
            "upper": upper.squeeze(-1),
        }


## 6. 可視化


In [ ]:
plt.figure(figsize=(9, 5))

for category in (0, 1, 2):
    mask = train_X[:, 1] == category
    result = posterior_by_category[category]

    plt.scatter(
        train_X[mask, 0],
        train_Y[mask, 0],
        label=f"observed category={category}",
    )
    plt.plot(
        grid,
        result["mean"],
        label=f"posterior category={category}",
    )

plt.xlabel("continuous x")
plt.ylabel("y")
plt.legend()
plt.show()


## 7. 混合変数のベイズ最適化

カテゴリ次元は連続最適化せず、`fixed_features_list` で各カテゴリ値を列挙します。BoTorchの `optimize_acqf_mixed` が各カテゴリ条件を考慮して候補を探索します。


In [ ]:
best_f = train_Y.max()
acqf = qLogExpectedImprovement(model=model, best_f=best_f)

bounds = torch.tensor(
    [
        [0.0, 0.0],
        [1.0, 2.0],
    ]
)

candidate, acq_value = optimize_acqf_mixed(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
    fixed_features_list=[
        {1: 0.0},
        {1: 1.0},
        {1: 2.0},
    ],
)

print("candidate:", candidate)
print("continuous value:", candidate[0, 0].item())
print("category:", int(candidate[0, 1].item()))
print("acquisition value:", acq_value)
print("objective(candidate):", objective(candidate))


## 8. 使い所

`MixedSingleTaskGP` は、温度 + 触媒種、圧力 + 装置種、組成比 + 製法など、連続条件とカテゴリ条件が混在する探索空間に適します。

カテゴリ変数を単純な連続値として `SingleTaskGP` に渡すのではなく、カテゴリ次元を `cat_dims` で明示してください。
